# vast.ai — Notebook 2: TRAIN on the RTX PRO 6000 (set-and-forget flywheel)

**Run all cells, then walk away.** It will:
1. start **Stage 1 breadth solving** in the background (CPU, fills the corpus),
2. run the **representation A/B** once (does colour-perm/D4 beat 0.149?),
3. enter the **ExIt flywheel** (Cell 4): harvest → train WM (gated, warm-start) →
   pretrain prior → repeat *forever*, auto-committing progress every 3 rounds.

The flywheel cell **blocks on purpose** (it's the work). The kernel keeps running
even if you close the browser tab. To stop: press ⏹ (interrupt) on Cell 4.
Code auto-scales to the card (batch 1024 @96GB, net_mult 4, AMP, TF32).

## Resume (pick up where it left off)
Everything is resumable: `solve_all` **skips already-solved levels** and resumes BFS from `/tmp/v19_frontier_*` checkpoints; `train_wm`/`pretrain` **warm-start** from the saved weights and keep only held-out improvements. So after Cell 0 (kill), Cells 2-4 continue the same corpus + model. If the instance died, a fresh box: Notebook 1 (clones the auto-pushed corpus+weights) -> Notebook 2 resumes from there.

In [ ]:
# Cell 0 — KILL any leftover runs from a previous session, THEN we resume.
# Orphaned background loops survive a kernel restart on Linux, so run this FIRST
# so you don't end up with duplicate breadth loops. It KEEPS the corpus, weights
# and frontier checkpoints, so the flywheel picks up exactly where it left off.
import subprocess
for pat in ["while true; do","solve_all.py","train_wm_v19","pretrain.py","harvest_wm","wm_augment"]:
    subprocess.run(f"pkill -f '{pat}'", shell=True)
subprocess.run("rm -f /tmp/v19_*.lock", shell=True)
print("killed leftover v19 runs; corpus + weights + frontier KEPT -> resumes")

In [ ]:
# Cell 1 — RTX setup (PY from Notebook 1)
import os, subprocess
PY  = open("/workspace/PY.txt").read().strip() if os.path.exists("/workspace/PY.txt") else __import__("sys").executable
V19 = "/workspace/arc3/CommunitySolutions/chronos_solver/v19"
print(subprocess.run([PY,"-c","import torch;torch.backends.cuda.matmul.allow_tf32=True;torch.backends.cudnn.allow_tf32=True;p=torch.cuda.get_device_properties(0);print('GPU',p.name,round(p.total_memory/1e9),'GB; TF32 on')"],capture_output=True,text=True).stdout)
has_flags = "net_mult" in open(f"{V19}/train_wm_v19.py").read()
assert has_flags, "box is missing the RTX flags -> run in a cell:  !git -C /workspace/arc3 pull"
print("PY =", PY, "| cores:", os.cpu_count(), "| RTX flags present: OK")

In [ ]:
# Cell 2 — STAGE 1: breadth solving in the background (CPU; fills the corpus)
import subprocess
subprocess.Popen(f"cd {V19} && while true; do {PY} solve_all.py --bfs-timeout 300 --shuffle; done",
                 shell=True, stdout=open(f"{V19}/solve_all.log","a"), stderr=subprocess.STDOUT)
print("Stage 1 breadth solving started in background (watch games count climb).")

In [ ]:
# Cell 3 — representation A/B (run ONCE): does colour-perm + D4 beat baseline 0.149?
!cd {V19} && {PY} wm_augment.py --epochs 40 --ncolor 4 --d4 2>&1 | grep augA

In [ ]:
# Cell 4 — THE FLYWHEEL (runs forever; interrupt to stop). harvest -> train (gated,
# warm-start) -> pretrain -> auto-commit every 3 rounds. The GPU keeps improving the
# model as Stage 1 grows breadth on CPU.
import subprocess, time, glob, json, re
DN = subprocess.DEVNULL
r = 0
while True:
    r += 1; t0 = time.time()
    subprocess.run(f"cd {V19} && {PY} harvest_wm.py", shell=True, stdout=DN, stderr=DN)
    subprocess.run(f"cd {V19} && {PY} train_wm_v19.py --epochs 40 --net-mult 4 --bsz -1 --amp --patience 8", shell=True, stdout=DN, stderr=DN)
    subprocess.run(f"cd {V19} && {PY} pretrain.py --net-mult 4 --bsz -1 --amp --per_game 1500 --epochs 30", shell=True, stdout=DN, stderr=DN)
    sols = glob.glob(f"{V19}/solutions/*.json"); games = len(sols)
    levels = sum(len(json.load(open(f))) for f in sols)
    try: best = re.findall(r"best HO chg-acc=([0-9.]+)", open(f"{V19}/WM_LOG.md").read())[-1]
    except Exception: best = "?"
    print(f"round {r}: games={games} levels={levels} WM_chg-acc={best} ({time.time()-t0:.0f}s)", flush=True)
    if r % 3 == 0:  # persist progress (needs GIT_TOKEN set in Notebook 1 Cell 1)
        subprocess.run("cd /workspace/arc3 && git add CommunitySolutions/chronos_solver/v19/*.pt "
                       "CommunitySolutions/chronos_solver/v19/*.md CommunitySolutions/chronos_solver/v19/solutions/ "
                       f"&& git commit -m 'rtx flywheel round {r}: {games} games {levels} levels' && git push origin main",
                       shell=True, stdout=DN, stderr=DN)
        print(f"  committed+pushed (round {r})", flush=True)
    time.sleep(30)

## Monitor / stop
- **While the flywheel runs**, Cell 4 prints a status line per round (`games`,
  `levels`, `WM_chg-acc`). That's your live monitor — scroll its output on check-back.
- For a second view, open a vast.ai **Terminal** (separate from the kernel) and:
  `tail -f /workspace/arc3/CommunitySolutions/chronos_solver/v19/WM_LOG.md`
- **Stop**: press ⏹ (interrupt) on Cell 4. Stage 1 keeps running until you also
  `pkill -f solve_all`.
- **Reading it**: chg-acc climbing as `games` grows → data-limited, keep going.
  Flat while games grow → architecture-bound (see WM_REPR_EXPERIMENT.md).